In [1]:
import os
import sys

repo_root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "part1" else os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from part1 import (
    gaussian_eliminate,
    back_substitution,
    determinant,
    inverse,
    rank_and_basis,
    verify_solution,
)

def fmt_obj(obj):
    if isinstance(obj, bool):
        return obj
    if isinstance(obj, int):
        return f"{obj:.4f}"
    if isinstance(obj, float):
        value = 0.0 if abs(obj) <= 1e-12 else obj
        return f"{value:.4f}"
    if isinstance(obj, list):
        return [fmt_obj(item) for item in obj]
    if isinstance(obj, tuple):
        return tuple(fmt_obj(item) for item in obj)
    if isinstance(obj, dict):
        return {key: fmt_obj(value) for key, value in obj.items()}
    return obj

# Part 1 Demo - Phép khử Gauss và các ứng dụng

Notebook này không chỉ chạy thử các hàm trong Part 1 mà còn tập trung vào những tình huống dễ quan sát bằng mắt khi học phép khử Gauss. Bốn case bên dưới lần lượt minh họa near-zero pivot để thấy partial pivoting hoạt động, hệ có vô số nghiệm, hệ vô nghiệm và một hệ ill-conditioned để cho thấy giới hạn số học của phương pháp.

## Case 1 - Near-zero pivot nhưng hệ vẫn giải tốt nhờ partial pivoting

Ở case này, phần tử đầu tiên của ma trận có độ lớn cỡ `10^-10`, nên nếu khử trực tiếp theo thứ tự dòng ban đầu thì việc chia cho pivot rất nhỏ sẽ dễ làm sai số tăng mạnh. Partial pivoting sẽ đổi dòng để chọn pivot lớn hơn, vì vậy hệ vẫn được giải ổn định và các hàm như `back_substitution`, `determinant`, `inverse` và `verify_solution` vẫn cho kết quả nhất quán.

In [2]:
A1 = [
    [1e-10, 2.34567, -1.23456],
    [3.5, -4.2, 1.125],
    [2.25, 1.33333, 5.75],
]
b1 = [1.2345, -2.3456, 3.4567]

ref1, info1, swap1 = gaussian_eliminate(A1, b1)

n1 = len(A1)
U1 = [row[:n1] for row in ref1[:n1]]
c1 = [row[n1] for row in ref1[:n1]]

x1_back = back_substitution(U1, c1)
det1 = determinant(A1)
inv1 = inverse(A1)
ver1 = verify_solution(A1, info1, b1)

print("REF augmented =", fmt_obj(ref1))
print("solution_info =", fmt_obj(info1))
print("swap_count =", swap1)
print("x from back_substitution =", fmt_obj(x1_back))
print("det(A1) =", fmt_obj(det1))
print("A1^-1 =", fmt_obj(inv1))
print("verification =", fmt_obj(ver1))

REF augmented = [['3.5000', '-4.2000', '1.1250', '-2.3456'], ['0.0000', '4.0333', '5.0268', '4.9646'], ['0.0000', '0.0000', '-4.1580', '-1.6528']]
solution_info = {'type': 'unique', 'x': ['0.0847', '0.7355', '0.3975'], 'pivot_columns': ['0.0000', '1.0000', '2.0000']}
swap_count = 2
x from back_substitution = ['0.0847', '0.7355', '0.3975']
det(A1) = -58.6970
A1^-1 = [['0.4370', '0.2578', '0.0434'], ['0.2997', '-0.0473', '0.0736'], ['-0.2405', '-0.0899', '0.1399']]
verification = {'is_close': True, 'residual_norm': '0.0000', 'relative_residual': '0.0000', 'numpy_solution': ['0.0847', '0.7355', '0.3975']}


Khi chạy cell trên, `swap_count` phải lớn hơn 0 vì partial pivoting đã đổi dòng để tránh pivot quá nhỏ. Nghiệm từ `back_substitution` và nghiệm trong `solution_info` phải trùng nhau, còn `residual_norm` và `relative_residual` nên rất nhỏ, cho thấy việc đổi dòng đã giúp hệ được giải ổn định.

## Case 2 - Hệ có vô số nghiệm

Case này dùng một ma trận phụ thuộc tuyến tính để minh họa nhánh `infinite`. Mục tiêu ở đây không phải tìm một nghiệm duy nhất, mà là quan sát cách chương trình trả về một nghiệm riêng, các biến tự do và cơ sở của null space.

In [3]:
A2 = [
    [1.2, 2.4, -0.6, 3.0],
    [2.4, 4.8, -1.2, 6.0],
    [0.6, 1.2, -0.3, 1.5],
]
b2 = [3.6, 7.2, 1.8]

ref2, info2, swap2 = gaussian_eliminate(A2, b2)
rb2 = rank_and_basis(A2)
ver2 = verify_solution(A2, info2, b2)

print("REF augmented =", fmt_obj(ref2))
print("solution_info =", fmt_obj(info2))
print("swap_count =", swap2)
print("rank_and_basis =", fmt_obj(rb2))
print("verification =", fmt_obj(ver2))

REF augmented = [['2.4000', '4.8000', '-1.2000', '6.0000', '7.2000'], ['0.0000', '0.0000', '0.0000', '0.0000', '0.0000'], ['0.0000', '0.0000', '0.0000', '0.0000', '0.0000']]
solution_info = {'type': 'infinite', 'particular': ['3.0000', '0.0000', '0.0000', '0.0000'], 'nullspace_basis': [['-2.0000', '1.0000', '0.0000', '0.0000'], ['0.5000', '0.0000', '1.0000', '0.0000'], ['-2.5000', '0.0000', '0.0000', '1.0000']], 'pivot_columns': ['0.0000'], 'free_columns': ['1.0000', '2.0000', '3.0000'], 'parameters': ['t1', 't2', 't3'], 'general_form': 'x = [3.0, 0.0, 0.0, 0.0] + t1*[-2.0, 1.0, 0.0, 0.0] + t2*[0.5, 0.0, 1.0, 0.0] + t3*[-2.5, 0.0, 0.0, 1.0]', 'rref_augmented': [['1.0000', '2.0000', '-0.5000', '2.5000', '3.0000'], ['0.0000', '0.0000', '0.0000', '0.0000', '0.0000'], ['0.0000', '0.0000', '0.0000', '0.0000', '0.0000']]}
swap_count = 1
rank_and_basis = {'rank': '1.0000', 'pivot_columns': ['0.0000'], 'free_columns': ['1.0000', '2.0000', '3.0000'], 'rref': [['1.0000', '2.0000', '-0.5000', '2.

Nếu chương trình phân loại đúng, `solution_info["type"]` sẽ là `infinite`, số pivot sẽ nhỏ hơn số ẩn và `rank_and_basis()` sẽ cho thấy null space có ít nhất một vector cơ sở. Ở nhánh này, hàm `verify_solution()` kiểm tra nghiệm riêng mà chương trình trả về, nên ta vẫn kỳ vọng residual nhỏ.

## Case 3 - Hệ vô nghiệm

Case này được chọn để sau khi khử Gauss xuất hiện một dòng có toàn bộ hệ số bằng 0 nhưng vế phải khác 0. Đây là dấu hiệu kinh điển cho thấy hệ phương trình không tương thích.

In [4]:
A3 = [
    [1, 1, 1],
    [2, 2, 2],
    [1, -1, 0],
]
b3 = [3, 7, 1]

ref3, info3, swap3 = gaussian_eliminate(A3, b3)

print("REF augmented =", fmt_obj(ref3))
print("solution_info =", fmt_obj(info3))
print("swap_count =", swap3)

REF augmented = [['2.0000', '2.0000', '2.0000', '7.0000'], ['0.0000', '-2.0000', '-1.0000', '-2.5000'], ['0.0000', '0.0000', '0.0000', '-0.5000']]
solution_info = {'type': 'inconsistent', 'pivot_columns': ['0.0000', '1.0000'], 'message': 'The system is inconsistent, so it has no solution.'}
swap_count = 2


Ở case này, `solution_info["type"]` phải là `inconsistent`. Vì hệ không có nghiệm nên ta chỉ dừng ở bước phân loại, không dùng `back_substitution`, `inverse` hay `verify_solution` để ép tạo ra một vector nghiệm không tồn tại.

## Case 4 - Hệ ill-conditioned và warning về pivot gần 0

Case cuối cùng dùng một ma trận gần suy biến. Hệ này vẫn có nghiệm duy nhất, nhưng trong quá trình khử sẽ xuất hiện pivot rất nhỏ so với quy mô của ma trận. Khi đó chương trình nên chèn warning `"pivot is near zero; the system may be ill-conditioned"` vào `solution_info` để báo rằng bài toán nhạy số và không nên tin tuyệt đối vào từng chữ số thập phân của nghiệm.

In [5]:
A4 = [
    [1.0, 1.0],
    [1.0, 1.0 + 1e-10],
]
b4 = [2.0, 2.0 + 1e-10]

ref4, info4, swap4 = gaussian_eliminate(A4, b4)
ver4 = verify_solution(A4, info4, b4)

print("REF augmented =", fmt_obj(ref4))
print("solution_info =", fmt_obj(info4))
print("swap_count =", swap4)
print("verification =", fmt_obj(ver4))

REF augmented = [['1.0000', '1.0000', '2.0000'], ['0.0000', '0.0000', '0.0000']]
solution_info = {'type': 'unique', 'x': ['1.0000', '1.0000'], 'pivot_columns': ['0.0000', '1.0000'], 'warnings': ['pivot is near zero; the system may be ill-conditioned']}
swap_count = 0
verification = {'is_close': True, 'residual_norm': '0.0000', 'relative_residual': '0.0000', 'numpy_solution': ['1.0000', '1.0000']}


Điểm cần theo dõi ở đây không phải là từng chữ số của nghiệm, mà là việc warning đã xuất hiện hay chưa và residual có còn nhỏ hay không. Nếu warning xuất hiện nhưng residual vẫn nhỏ, ta có thể kết luận rằng chương trình đã nhận diện đúng rủi ro tính toán số, đồng thời nghiệm trả về vẫn nhất quán với hệ ở mức sai số chấp nhận được.